In [6]:
import json
from pathlib import Path
import random

# Dataset path
VAL_DATASET_DIR = Path("/datastor1/dy4652/proteinzen/geom_drugs_conformers/val")
VAL_MANIFEST_PATH = VAL_DATASET_DIR / "manifest.json"

In [7]:
with open(VAL_MANIFEST_PATH, 'r') as f:
    val_manifest = json.load(f)

print(f"Total entries in val manifest: {len(val_manifest)}")

Total entries in val manifest: 539


In [8]:
import numpy as np

def get_npz_paths(entry_ids, dataset_dir):
    """Get paths to .npz files for all conformers of one molecule."""
    paths = []
    for sid in entry_ids:
        mid = sid[1:3] if len(sid) > 2 else "00"
        paths.append(dataset_dir / "structures" / mid / f"{sid}.npz")
    return paths

def bond_angle_rad(pa, pb, pc):
    """Bond angle at pb (radians), shape (..., 3)."""
    u = np.asarray(pa) - np.asarray(pb)
    v = np.asarray(pc) - np.asarray(pb)
    nru = np.linalg.norm(u, axis=-1, keepdims=True)
    nrv = np.linalg.norm(v, axis=-1, keepdims=True)
    u = u / (nru + 1e-10)
    v = v / (nrv + 1e-10)
    return np.arccos(np.clip(np.sum(u * v, axis=-1), -1.0, 1.0))

def torsion_angle_rad(p1, p2, p3, p4):
    """Dihedral angle p1-p2-p3-p4 in radians."""
    p1, p2, p3, p4 = [np.asarray(x) for x in (p1, p2, p3, p4)]
    b1 = p2 - p1
    b2 = p3 - p2
    b3 = p4 - p3
    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)
    nb2 = np.linalg.norm(b2, axis=-1, keepdims=True) + 1e-10
    n1 = n1 / (np.linalg.norm(n1, axis=-1, keepdims=True) + 1e-10)
    n2 = n2 / (np.linalg.norm(n2, axis=-1, keepdims=True) + 1e-10)
    m1 = np.cross(n1, b2 / nb2)
    return np.arctan2(np.sum(m1 * n2, axis=-1), np.sum(n1 * n2, axis=-1) + 1e-10)

def build_neighbors(bonds, n_atoms):
    """bonds: (N, 2) atom_1, atom_2. Returns list of sets of neighbors per atom."""
    neighbors = [set() for _ in range(n_atoms)]
    for i, j in bonds:
        if i < n_atoms and j < n_atoms:
            neighbors[i].add(j)
            neighbors[j].add(i)
    return neighbors

def angle_triples_and_torsion_quads(neighbors):
    """From neighbor list, return list of (a,b,c) for bond angles and (a,b,c,d) for torsions."""
    triples = []
    quads = []
    for b in range(len(neighbors)):
        nb = list(neighbors[b])
        for i, a in enumerate(nb):
            for c in nb[i+1:]:
                triples.append((a, b, c))
    for b in range(len(neighbors)):
        for c in neighbors[b]:
            if c <= b:
                continue
            for a in neighbors[b]:
                if a == c:
                    continue
                for d in neighbors[c]:
                    if d == b:
                        continue
                    quads.append((a, b, c, d))
    return triples, quads

def load_conformers_npz(paths):
    """Load coords and bonds from list of npz paths. Returns (list of coords arrays, bonds, n_atoms)."""
    all_coords = []
    bonds = None
    n_atoms = None
    for p in paths:
        if not p.exists():
            return None, None, None
        data = np.load(p, allow_pickle=True)
        atoms = data["atoms"]
        coords = np.asarray(atoms["coords"], dtype=np.float64)
        b = data["bonds"]
        bond_arr = np.column_stack([b["atom_1"], b["atom_2"]])
        if bonds is None:
            bonds = bond_arr
            n_atoms = len(coords)
        all_coords.append(coords)
    return all_coords, bonds, n_atoms

def mean_angular_change_rad(angles_per_conformer):
    """angles_per_conformer: (n_conformers, n_angles). Return mean abs change from mean angle (in rad)."""
    arr = np.asarray(angles_per_conformer)
    if arr.size == 0:
        return np.nan
    mean_angle = np.nanmean(arr, axis=0)
    diff = np.abs(arr - mean_angle)
    # wrap to [-pi, pi] for circular difference
    diff = np.minimum(diff, 2 * np.pi - diff)
    return np.nanmean(diff)

def bond_lengths_for_coords(coords, bonds):
    """bonds: (N_bonds, 2) atom indices. Return 1d array of bond lengths in Angstroms."""
    lengths = []
    for (i, j) in bonds:
        d = np.linalg.norm(np.asarray(coords[i]) - np.asarray(coords[j]))
        lengths.append(d)
    return np.array(lengths)

def mean_bond_length_change_angstrom(lengths_per_conformer):
    """lengths_per_conformer: (n_conformers, n_bonds). Return mean abs change from mean length (in Angstroms)."""
    arr = np.asarray(lengths_per_conformer)
    if arr.size == 0:
        return np.nan
    mean_len = np.nanmean(arr, axis=0)
    diff = np.abs(arr - mean_len)
    return np.nanmean(diff)

def conformer_angle_stats_for_entry(entry, dataset_dir):
    """For one manifest entry, load conformers, compute per-angle/length change across conformers, then average."""
    ids = entry.get("ids")
    if not ids or len(ids) < 2:
        return np.nan, np.nan, np.nan, 0
    paths = get_npz_paths(ids, dataset_dir)
    all_coords, bonds, n_atoms = load_conformers_npz(paths)
    if all_coords is None or len(all_coords) < 2:
        return np.nan, np.nan, np.nan, 0
    neighbors = build_neighbors(bonds, n_atoms)
    triples, quads = angle_triples_and_torsion_quads(neighbors)
    bond_angles_per_conf = []
    torsion_angles_per_conf = []
    bond_lengths_per_conf = []
    for coords in all_coords:
        ba = np.array([bond_angle_rad(coords[a], coords[b], coords[c]) for a, b, c in triples])
        ta = np.array([torsion_angle_rad(coords[a], coords[b], coords[c], coords[d]) for a, b, c, d in quads])
        bl = bond_lengths_for_coords(coords, bonds)
        bond_angles_per_conf.append(ba)
        torsion_angles_per_conf.append(ta)
        bond_lengths_per_conf.append(bl)
    bond_change = mean_angular_change_rad(bond_angles_per_conf)
    torsion_change = mean_angular_change_rad(torsion_angles_per_conf)
    bond_length_change = mean_bond_length_change_angstrom(bond_lengths_per_conf)
    n_mol = 1
    return bond_change, torsion_change, bond_length_change, n_mol

def aggregate_over_manifests(manifests_and_dirs):
    """
    manifests_and_dirs: list of (manifest_list, dataset_dir Path).
    Returns: mean bond angle change (rad), mean torsion angle change (rad), mean bond length change (Angstrom),
             bond_deg, torsion_deg, n_bond, n_torsion, n_bond_length.
    """
    all_bond_changes = []
    all_torsion_changes = []
    all_bond_length_changes = []
    for manifest, dataset_dir in manifests_and_dirs:
        for entry in manifest:
            try:
                b, t, bl, _ = conformer_angle_stats_for_entry(entry, dataset_dir)
                if not np.isnan(b):
                    all_bond_changes.append(b)
                if not np.isnan(t):
                    all_torsion_changes.append(t)
                if not np.isnan(bl):
                    all_bond_length_changes.append(bl)
            except Exception as e:
                continue
    bond_rad = np.mean(all_bond_changes) if all_bond_changes else np.nan
    torsion_rad = np.mean(all_torsion_changes) if all_torsion_changes else np.nan
    bond_length_angstrom = np.mean(all_bond_length_changes) if all_bond_length_changes else np.nan
    bond_deg = np.degrees(bond_rad) if not np.isnan(bond_rad) else np.nan
    torsion_deg = np.degrees(torsion_rad) if not np.isnan(torsion_rad) else np.nan
    return bond_rad, torsion_rad, bond_length_angstrom, bond_deg, torsion_deg, len(all_bond_changes), len(all_torsion_changes), len(all_bond_length_changes)

In [9]:
# Run on val manifest: average change in bond angles, torsion angles, and bond lengths
# First average across all angles/lengths per molecule, then average across molecules
manifests_and_dirs = [(val_manifest, VAL_DATASET_DIR)]
bond_rad, torsion_rad, bond_length_angstrom, bond_deg, torsion_deg, n_bond, n_torsion, n_bond_length = aggregate_over_manifests(manifests_and_dirs)

print("Val manifest — average change across conformers")
print(f"  Molecules with ≥2 conformers: bond_angles={n_bond}, torsion_angles={n_torsion}, bond_lengths={n_bond_length}")
print(f"  Mean bond angle change:    {bond_deg:.4f}° ({bond_rad:.6f} rad)")
print(f"  Mean torsion angle change: {torsion_deg:.4f}° ({torsion_rad:.6f} rad)")
print(f"  Mean bond length change:  {bond_length_angstrom:.6f} Å")

Val manifest — average change across conformers
  Molecules with ≥2 conformers: bond_angles=511, torsion_angles=511, bond_lengths=511
  Mean bond angle change:    0.5469° (0.009545 rad)
  Mean torsion angle change: 61.1289° (1.066900 rad)
  Mean bond length change:  0.001494 Å


In [10]:
# Optional: run on multiple manifests (e.g. train, val, test) and average across all
# Uncomment and set paths to include train/test:
# TRAIN_DATASET_DIR = Path("/datastor1/dy4652/proteinzen/geom_drugs_conformers/train")
# TEST_DATASET_DIR = Path("/datastor1/dy4652/proteinzen/geom_drugs_conformers/test")
# with open(TRAIN_DATASET_DIR / "manifest.json") as f: train_manifest = json.load(f)
# with open(TEST_DATASET_DIR / "manifest.json") as f: test_manifest = json.load(f)
# manifests_and_dirs = [(val_manifest, VAL_DATASET_DIR), (train_manifest, TRAIN_DATASET_DIR), (test_manifest, TEST_DATASET_DIR)]
# bond_rad, torsion_rad, bond_length_angstrom, bond_deg, torsion_deg, n_bond, n_torsion, n_bond_length = aggregate_over_manifests(manifests_and_dirs)
# print("All manifests — mean bond angle (deg):", bond_deg, "torsion (deg):", torsion_deg, "bond length (Å):", bond_length_angstrom)

results = {"bond_deg": bond_deg, "torsion_deg": torsion_deg, "bond_length_angstrom": bond_length_angstrom, "n_molecules_bond": n_bond, "n_molecules_torsion": n_torsion, "n_molecules_bond_length": n_bond_length}
results

{'bond_deg': np.float64(0.5468643015144541),
 'torsion_deg': np.float64(61.12885871046869),
 'bond_length_angstrom': np.float64(0.0014935203212636543),
 'n_molecules_bond': 511,
 'n_molecules_torsion': 511,
 'n_molecules_bond_length': 511}